Subir el JSON

In [ ]:
from google.colab import files
print("Subí tu training_data.json")
uploaded = files.upload()


Descargar desde google drive

In [ ]:
!pip install gdown -q
import gdown
import re

url = ""

match = re.search(r"(?:/d/|id=)([a-zA-Z0-9_-]+)", url)
file_id = match.group(1)
gdown.download(f"https://drive.google.com/uc?id={file_id}", "training_data.json", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1v0QmZs9-r1xtE4d3xXkqwHssq8dNReYr
From (redirected): https://drive.google.com/uc?id=1v0QmZs9-r1xtE4d3xXkqwHssq8dNReYr&confirm=t&uuid=441ffba4-446e-4dcd-91b7-fbb85dcfaffd
To: /content/training_data.json
100%|██████████| 120M/120M [00:01<00:00, 88.4MB/s]


'training_data.json'

Convertir a dataset YOLO


In [ ]:
import json, base64, random, hashlib
from pathlib import Path
from collections import defaultdict


def _folder_hash(folder: str) -> str:
    return hashlib.sha1(folder.encode()).hexdigest()[:6]


def unique_name(image_name: str, folder: str) -> str:
    stem, suffix = Path(image_name).stem, Path(image_name).suffix
    return f"{stem}_{_folder_hash(folder)}{suffix}"

TRAINING_JSON = Path('training_data.json')
DATASET_DIR = Path('dataset')
TRAIN_RATIO = 0.8

data = json.loads(TRAINING_JSON.read_text(encoding='utf-8'))
print(f"Total muestras: {len(data)}")

classes = sorted({e['class_type'] for e in data})
cls_to_id = {c: i for i, c in enumerate(classes)}
print(f"Clases ({len(classes)}): {classes}")

counts = defaultdict(int)
for e in data:
    counts[e['class_type']] += 1
for c in classes:
    print(f"  {c}: {counts[c]}")

by_image = defaultdict(list)
for e in data:
    key = (e['watermark_folder'], e['image_name'])
    by_image[key].append(e)

image_keys = sorted(by_image.keys())
random.seed(42)
random.shuffle(image_keys)
n_train = int(len(image_keys) * TRAIN_RATIO)
train_set = set(image_keys[:n_train])

for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (DATASET_DIR / sub).mkdir(parents=True, exist_ok=True)

for (folder, img_name), entries in by_image.items():
    split = 'train' if (folder, img_name) in train_set else 'val'
    out_name = unique_name(img_name, folder)
    img_bytes = base64.b64decode(entries[0]['image_base64'])
    (DATASET_DIR / 'images' / split / out_name).write_bytes(img_bytes)

    lines = []
    for e in entries:
        cls_id = cls_to_id[e['class_type']]
        cx, cy, bw, bh = e['bbox_yolo']
        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    label_path = DATASET_DIR / 'labels' / split / f"{Path(out_name).stem}.txt"
    label_path.write_text("\n".join(lines) + "\n", encoding='utf-8')

# dataset.yaml
yaml_lines = [
    f"path: {DATASET_DIR.absolute().as_posix()}",
    "train: images/train",
    "val: images/val",
    "",
    "names:",
]
for i, c in enumerate(classes):
    yaml_lines.append(f"  {i}: {c}")

(DATASET_DIR / 'dataset.yaml').write_text("\n".join(yaml_lines), encoding='utf-8')
print(f"\n✅ Dataset listo: {DATASET_DIR.absolute()}")

Total muestras: 1291
Clases (5): ['alto', 'apunta', 'color', 'gris', 'texto']
  alto: 89
  apunta: 72
  color: 537
  gris: 521
  texto: 72

✅ Dataset listo: /content/dataset


Activar GPU y entrenar

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="dataset/dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="watermark_detector",
    patience=20,
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fracti

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f1d49551820>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [ ]:
from google.colab import files
files.download('runs/detect/watermark_detector/weights/best.pt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>